# Observational Methods and CATE Comparison

**Session 129 Tutorial**

This notebook compares estimators on the same data:

1. **Part 1**: Observational study methods (PSM, IPW, DR, TMLE)
2. **Part 2**: Heterogeneous treatment effect methods (S/T/X/R-learners, DML, Causal Forest)

## Learning Objectives

- Compare observational ATE estimators on identical data
- Understand double robustness advantage
- Evaluate CATE methods by correlation with true effects
- Visualize treatment effect heterogeneity

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

np.random.seed(42)
print("Setup complete")

In [ ]:
# Import estimators
import sys
sys.path.insert(0, '/home/brandon_behring/Claude/causal_inference_mastery')

# Observational methods
from src.causal_inference.psm import psm_ate
from src.causal_inference.observational import (
    ipw_ate_observational, 
    dr_ate, 
    tmle_ate,
    estimate_propensity,
)

# CATE methods
from src.causal_inference.cate import (
    s_learner,
    t_learner,
    x_learner,
    r_learner,
    double_ml,
    causal_forest,
)

print("All estimators loaded successfully")

---
# Part 1: Observational Study Comparison

## Methods Being Compared

| Method | Model Required | Double Robust? |
|--------|---------------|----------------|
| PSM | Propensity only | No |
| IPW | Propensity only | No |
| DR (AIPW) | Both propensity + outcome | **Yes** |
| TMLE | Both + targeting | **Yes** |

## Double Robustness

DR and TMLE are consistent if **either**:
- Propensity score model is correctly specified, OR
- Outcome model is correctly specified

This provides insurance against model misspecification.

In [ ]:
# Data generating process for observational study
def generate_observational_dgp(
    n: int = 2000,
    true_ate: float = 2.0,
    confounding_strength: float = 0.8,
    seed: int = 42,
) -> dict:
    """Generate observational data with confounding.
    
    Model:
        X ~ N(0, I_5)
        propensity = expit(confounding * X[:,0] + 0.3*X[:,1])
        D ~ Bernoulli(propensity)
        Y = 1 + X[:,0] + 0.5*X[:,1] + ATE*D + noise
        
    The confounder X[:,0] affects both treatment AND outcome,
    creating bias if not adjusted for.
    """
    rng = np.random.default_rng(seed)
    
    # Covariates
    X = rng.normal(0, 1, (n, 5))
    
    # Propensity score (confounded by X[:,0] and X[:,1])
    logit_p = confounding_strength * X[:, 0] + 0.3 * X[:, 1]
    propensity = 1 / (1 + np.exp(-logit_p))
    
    # Treatment assignment
    D = rng.binomial(1, propensity, n)
    
    # Outcome (confounded by X[:,0] and X[:,1])
    baseline = 1 + X[:, 0] + 0.5 * X[:, 1]
    noise = rng.normal(0, 1, n)
    Y = baseline + true_ate * D + noise
    
    return {
        'outcome': Y,
        'treatment': D,
        'covariates': X,
        'propensity': propensity,
        'true_ate': true_ate,
    }

obs_data = generate_observational_dgp(n=2000, true_ate=2.0, confounding_strength=0.8)
print(f"Generated {len(obs_data['outcome'])} observations")
print(f"Treatment rate: {obs_data['treatment'].mean():.1%}")
print(f"True ATE: {obs_data['true_ate']}")

In [ ]:
# Visualize propensity score overlap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

Y = obs_data['outcome']
D = obs_data['treatment']
X = obs_data['covariates']

# Propensity score overlap
ax = axes[0]
ax.hist(obs_data['propensity'][D == 0], bins=40, alpha=0.6, 
        label=f'Control (n={np.sum(D==0)})', density=True, color='steelblue')
ax.hist(obs_data['propensity'][D == 1], bins=40, alpha=0.6, 
        label=f'Treated (n={np.sum(D==1)})', density=True, color='coral')
ax.set_xlabel('True Propensity Score')
ax.set_ylabel('Density')
ax.set_title('Propensity Score Overlap')
ax.legend()
ax.axvline(0.1, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.9, color='gray', linestyle='--', alpha=0.5)

# SMD before adjustment
ax = axes[1]
smds = []
for j in range(X.shape[1]):
    treated_mean = X[D == 1, j].mean()
    control_mean = X[D == 0, j].mean()
    pooled_var = (X[D == 1, j].var() + X[D == 0, j].var()) / 2
    smd = (treated_mean - control_mean) / np.sqrt(pooled_var)
    smds.append(smd)

ax.barh(range(X.shape[1]), smds, color='steelblue', alpha=0.7)
ax.axvline(0.1, color='green', linestyle='--', label='|SMD| < 0.1 (good)')
ax.axvline(-0.1, color='green', linestyle='--')
ax.axvline(0.25, color='red', linestyle='--', label='|SMD| > 0.25 (poor)')
ax.axvline(-0.25, color='red', linestyle='--')
ax.set_yticks(range(X.shape[1]))
ax.set_yticklabels([f'X[{j}]' for j in range(X.shape[1])])
ax.set_xlabel('Standardized Mean Difference')
ax.set_title('Covariate Balance (Before Adjustment)')
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

print(f"\nX[0] is the primary confounder: SMD = {smds[0]:.3f}")

In [ ]:
# Run all observational methods
obs_results = {}

# 1. PSM
print("Running PSM...")
psm_result = psm_ate(Y, D, X)
obs_results['PSM'] = {
    'estimate': psm_result['estimate'],
    'se': psm_result['se'],
    'model': 'Propensity only',
}

# 2. IPW
print("Running IPW...")
ipw_result = ipw_ate_observational(Y, D, X)
obs_results['IPW'] = {
    'estimate': ipw_result['estimate'],
    'se': ipw_result['se'],
    'model': 'Propensity only',
}

# 3. DR (AIPW)
print("Running DR (AIPW)...")
dr_result = dr_ate(Y, D, X)
obs_results['DR (AIPW)'] = {
    'estimate': dr_result['estimate'],
    'se': dr_result['se'],
    'model': 'Both (double robust)',
}

# 4. TMLE
print("Running TMLE...")
tmle_result = tmle_ate(Y, D, X)
obs_results['TMLE'] = {
    'estimate': tmle_result['estimate'],
    'se': tmle_result['se'],
    'model': 'Both + targeting',
}

print("\nAll methods complete!")

In [ ]:
# Display comparison table
true_ate = obs_data['true_ate']

print(f"True ATE: {true_ate:.3f}")
print("\n" + "="*80)
print(f"{'Method':<12} {'Estimate':>10} {'SE':>10} {'Bias':>10} {'95% CI':>20} {'Covers':>10}")
print("="*80)

for name, res in obs_results.items():
    est = res['estimate']
    se = res['se']
    bias = est - true_ate
    ci_lower = est - 1.96 * se
    ci_upper = est + 1.96 * se
    covers = ci_lower < true_ate < ci_upper
    ci_str = f"[{ci_lower:.2f}, {ci_upper:.2f}]"
    print(f"{name:<12} {est:>10.3f} {se:>10.3f} {bias:>+10.3f} {ci_str:>20} {str(covers):>10}")

In [ ]:
# Forest plot
fig, ax = plt.subplots(figsize=(10, 5))

methods = list(obs_results.keys())
y_pos = np.arange(len(methods))
estimates = [obs_results[m]['estimate'] for m in methods]
ses = [obs_results[m]['se'] for m in methods]

# Color by double robustness
colors = ['steelblue', 'steelblue', 'forestgreen', 'forestgreen']

ax.errorbar(estimates, y_pos, xerr=[1.96*s for s in ses], fmt='o', 
            capsize=5, markersize=10, capthick=2, color='black')

# Color points
for i, (est, color) in enumerate(zip(estimates, colors)):
    ax.plot(est, i, 'o', markersize=10, color=color)

# True ATE
ax.axvline(x=true_ate, color='red', linestyle='--', linewidth=2, 
           label=f'True ATE = {true_ate}')

ax.set_yticks(y_pos)
ax.set_yticklabels(methods)
ax.set_xlabel('Average Treatment Effect (95% CI)')
ax.set_title('Observational Estimator Comparison')

# Legend for colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Single robust'),
    Patch(facecolor='forestgreen', label='Doubly robust'),
    plt.Line2D([0], [0], color='red', linestyle='--', label=f'True ATE = {true_ate}')
]
ax.legend(handles=legend_elements, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Observational Methods: Key Takeaways

1. **All methods** can recover the true ATE when models are correctly specified
2. **DR and TMLE** provide protection against single-model misspecification
3. **PSM** may have higher variance due to matching
4. **IPW** can be unstable with extreme propensity scores

### Recommendations

| Scenario | Recommended Method |
|----------|-------------------|
| Confident in propensity model | IPW |
| Limited overlap | PSM (focuses on common support) |
| Uncertain about models | DR or TMLE |
| Targeting specific estimand | TMLE |

---
# Part 2: CATE Method Comparison

## Methods Being Compared

| Method | Approach | Strengths |
|--------|----------|----------|
| S-learner | Single model with treatment as feature | Simple |
| T-learner | Separate models per treatment group | Intuitive |
| X-learner | Cross-fitting with imputation | Handles imbalance |
| R-learner | Robinson transformation | Doubly robust |
| Double ML | Cross-fitted R-learner | Debiased, valid CI |
| Causal Forest | Honest random forest | Nonlinear heterogeneity |

## Evaluation Metrics

1. **ATE Recovery**: Mean(CATE) vs True ATE
2. **CATE Correlation**: Pearson r between estimated and true CATE
3. **CATE RMSE**: Root mean squared error of CATE estimates

In [ ]:
# Data generating process with heterogeneous effects
def generate_heterogeneous_dgp(
    n: int = 2000,
    p: int = 5,
    true_ate: float = 2.0,
    heterogeneity: float = 1.0,
    seed: int = 42,
) -> dict:
    """Generate data with treatment effect heterogeneity.
    
    Model:
        X ~ N(0, I_p)
        tau(X) = true_ate + heterogeneity * X[:,0]  # Linear heterogeneity
        D ~ Bernoulli(0.5)  # Randomized
        Y = baseline(X) + tau(X)*D + noise
        
    Returns true CATE tau(X) for validation.
    """
    rng = np.random.default_rng(seed)
    
    # Covariates
    X = rng.normal(0, 1, (n, p))
    
    # Treatment (randomized for clean comparison)
    D = rng.binomial(1, 0.5, n)
    
    # Heterogeneous treatment effect: tau(X) = ATE + heterogeneity * X[:,0]
    true_cate = true_ate + heterogeneity * X[:, 0]
    
    # Baseline outcome
    baseline = 1 + 0.5 * X[:, 0] + 0.3 * X[:, 1]
    
    # Outcome
    noise = rng.normal(0, 1, n)
    Y = baseline + true_cate * D + noise
    
    return {
        'outcome': Y,
        'treatment': D,
        'covariates': X,
        'true_cate': true_cate,
        'true_ate': np.mean(true_cate),
    }

cate_data = generate_heterogeneous_dgp(n=2000, true_ate=2.0, heterogeneity=1.0)
print(f"Generated {len(cate_data['outcome'])} observations")
print(f"True ATE: {cate_data['true_ate']:.3f}")
print(f"True CATE range: [{cate_data['true_cate'].min():.2f}, {cate_data['true_cate'].max():.2f}]")
print(f"True CATE std: {cate_data['true_cate'].std():.3f}")

In [ ]:
# Visualize true heterogeneity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CATE distribution
ax = axes[0]
ax.hist(cate_data['true_cate'], bins=40, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(cate_data['true_ate'], color='red', linestyle='--', linewidth=2, 
           label=f'Mean (ATE) = {cate_data["true_ate"]:.2f}')
ax.set_xlabel('True CATE')
ax.set_ylabel('Count')
ax.set_title('Distribution of True Treatment Effects')
ax.legend()

# CATE vs X[0]
ax = axes[1]
X_cate = cate_data['covariates']
ax.scatter(X_cate[:, 0], cate_data['true_cate'], alpha=0.3, s=10)
# Fit line
z = np.polyfit(X_cate[:, 0], cate_data['true_cate'], 1)
x_line = np.linspace(X_cate[:, 0].min(), X_cate[:, 0].max(), 100)
ax.plot(x_line, np.polyval(z, x_line), 'r-', linewidth=2, 
        label=f'τ(X) = {z[1]:.1f} + {z[0]:.1f}×X[0]')
ax.set_xlabel('X[0] (Effect Modifier)')
ax.set_ylabel('True CATE')
ax.set_title('Treatment Effect Heterogeneity')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Run all CATE methods
Y_cate = cate_data['outcome']
D_cate = cate_data['treatment']
X_cate = cate_data['covariates']
true_cate = cate_data['true_cate']
true_ate_cate = cate_data['true_ate']

cate_results = {}

print("Running CATE methods...")

# S-learner
print("  S-learner...")
s_result = s_learner(outcomes=Y_cate, treatment=D_cate, covariates=X_cate, model='linear')
cate_results['S-learner'] = s_result

# T-learner
print("  T-learner...")
t_result = t_learner(outcomes=Y_cate, treatment=D_cate, covariates=X_cate, model='linear')
cate_results['T-learner'] = t_result

# X-learner
print("  X-learner...")
x_result = x_learner(outcomes=Y_cate, treatment=D_cate, covariates=X_cate, model='linear')
cate_results['X-learner'] = x_result

# R-learner
print("  R-learner...")
r_result = r_learner(outcomes=Y_cate, treatment=D_cate, covariates=X_cate, model='linear')
cate_results['R-learner'] = r_result

# Double ML
print("  Double ML...")
dml_result = double_ml(outcomes=Y_cate, treatment=D_cate, covariates=X_cate)
cate_results['Double ML'] = dml_result

# Causal Forest
print("  Causal Forest...")
cf_result = causal_forest(outcomes=Y_cate, treatment=D_cate, covariates=X_cate)
cate_results['Causal Forest'] = cf_result

print("\nAll CATE methods complete!")

In [ ]:
# Compute evaluation metrics
metrics = {}

for name, result in cate_results.items():
    est_cate = result['cate']
    est_ate = result['ate']
    
    # ATE recovery
    ate_bias = est_ate - true_ate_cate
    
    # CATE correlation
    correlation, p_value = stats.pearsonr(est_cate, true_cate)
    
    # CATE RMSE
    rmse = np.sqrt(np.mean((est_cate - true_cate)**2))
    
    metrics[name] = {
        'ate': est_ate,
        'ate_bias': ate_bias,
        'correlation': correlation,
        'rmse': rmse,
    }

# Display metrics table
print(f"True ATE: {true_ate_cate:.3f}")
print("\n" + "="*75)
print(f"{'Method':<15} {'ATE':>8} {'ATE Bias':>10} {'CATE Corr':>12} {'CATE RMSE':>12}")
print("="*75)

for name, m in metrics.items():
    print(f"{name:<15} {m['ate']:>8.3f} {m['ate_bias']:>+10.3f} {m['correlation']:>12.3f} {m['rmse']:>12.3f}")

In [ ]:
# Scatter plots: Estimated CATE vs True CATE
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, result) in enumerate(cate_results.items()):
    ax = axes[idx]
    est_cate = result['cate']
    
    ax.scatter(true_cate, est_cate, alpha=0.3, s=10)
    
    # Perfect prediction line
    min_val = min(true_cate.min(), est_cate.min())
    max_val = max(true_cate.max(), est_cate.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
    
    # Fit line
    z = np.polyfit(true_cate, est_cate, 1)
    ax.plot([min_val, max_val], np.polyval(z, [min_val, max_val]), 'g-', 
            linewidth=2, alpha=0.7, label=f'Fit (r={metrics[name]["correlation"]:.2f})')
    
    ax.set_xlabel('True CATE')
    ax.set_ylabel('Estimated CATE')
    ax.set_title(name)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CATE distributions comparison
fig, ax = plt.subplots(figsize=(12, 6))

# Violin plot data
positions = np.arange(len(cate_results) + 1)
data_for_violin = [true_cate] + [result['cate'] for result in cate_results.values()]
labels = ['True'] + list(cate_results.keys())

parts = ax.violinplot(data_for_violin, positions=positions, showmeans=True, showmedians=True)

# Color the true distribution differently
parts['bodies'][0].set_facecolor('lightcoral')
parts['bodies'][0].set_alpha(0.7)
for i in range(1, len(parts['bodies'])):
    parts['bodies'][i].set_facecolor('steelblue')
    parts['bodies'][i].set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('CATE')
ax.set_title('CATE Distribution Comparison')
ax.axhline(true_ate_cate, color='red', linestyle='--', label=f'True ATE = {true_ate_cate:.2f}')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: Correlation comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation
ax = axes[0]
correlations = [metrics[m]['correlation'] for m in cate_results.keys()]
bars = ax.bar(list(cate_results.keys()), correlations, color='steelblue', alpha=0.7, edgecolor='black')
ax.axhline(0.9, color='green', linestyle='--', label='r = 0.9 (excellent)')
ax.axhline(0.7, color='orange', linestyle='--', label='r = 0.7 (good)')
ax.set_ylabel('Correlation with True CATE')
ax.set_title('CATE Recovery: Correlation')
ax.legend()
ax.set_ylim(0, 1.05)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# RMSE
ax = axes[1]
rmses = [metrics[m]['rmse'] for m in cate_results.keys()]
bars = ax.bar(list(cate_results.keys()), rmses, color='coral', alpha=0.7, edgecolor='black')
ax.set_ylabel('RMSE')
ax.set_title('CATE Recovery: RMSE (lower is better)')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## CATE Methods: Key Takeaways

### Linear Heterogeneity (This Example)

With linear treatment effect heterogeneity (τ(X) = β₀ + β₁X), **T-learner** and **R-learner** typically perform well because they naturally capture linear interactions.

### Method Selection Guidelines

| Scenario | Recommended Methods |
|----------|--------------------|
| Linear heterogeneity | T-learner, R-learner, Double ML |
| Nonlinear heterogeneity | Causal Forest, X-learner with RF |
| Small sample size | S-learner, T-learner |
| Valid inference needed | Double ML (has valid CIs) |
| Imbalanced treatment groups | X-learner |

### Limitations

- **S-learner**: Biased toward zero (regularization shrinks treatment coefficient)
- **T-learner**: Extrapolation issues in regions with poor overlap
- **Causal Forest**: May not detect linear heterogeneity well

---
# Summary

## Part 1: Observational Methods

- **PSM, IPW**: Work well with correct propensity model
- **DR, TMLE**: Double robustness provides insurance
- **Recommendation**: Use DR/TMLE when model uncertainty exists

## Part 2: CATE Methods

- All methods recover ATE reasonably well
- **T-learner, R-learner, DML**: Best for linear heterogeneity
- **Causal Forest**: Best for complex nonlinear patterns
- **S-learner**: Simple but biased toward homogeneous effects

## References

- Robins JM, Rotnitzky A, Zhao LP (1994). AIPW estimation.
- van der Laan MJ, Rubin DB (2006). TMLE.
- Künzel SR, et al. (2019). Meta-learners for CATE.
- Wager S, Athey S (2018). Causal forests.
- Chernozhukov V, et al. (2018). Double/debiased ML.